# Day12 actual-video pilot

Reconstructed continuation notebook. Saved guide evidence is preserved; a completed GPU video result is not implied. Select a CUDA runtime (T4), read PROTOCOL.md, and run cells in order. Use the separate pinned checkout below; an existing checkout is verified and never reset. Authentication is handled by the user/secure browser sign-in, not notebook credentials.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Fail before downloads if this runtime has no usable CUDA GPU.
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c', 'import torch; assert torch.cuda.is_available(), "CUDA runtime required"'], check=True)
repo = Path('/content/vjepa-day12')
revision = '5c4bb45870e303f9756c9bfe6fa1782147b6dfd7'
if not repo.exists():
    subprocess.run(['git', 'clone', '--no-checkout', '--filter=blob:none', '--single-branch', '--branch', 'experiment/part-consistency-colab', 'https://github.com/skbdps/vjepa-poc.git', str(repo)], check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', '--no-cone', '/experiments/day11/requirements.txt', '/experiments/day11/vace_bridge.py', '/experiments/day12/'], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', '--detach', revision], cwd=repo, check=True)
actual_revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, text=True, capture_output=True).stdout.strip()
local_changes = subprocess.run(['git', 'status', '--porcelain'], cwd=repo, check=True, text=True, capture_output=True).stdout.strip()
if actual_revision != revision or local_changes:
    raise RuntimeError('Existing checkout differs from the fixed pilot; preserve it and use a fresh path. Nothing was reset.')

def run(*arguments):
    subprocess.run([sys.executable, *arguments], cwd=repo, check=True)

print('Verified fixed checkout:', repo, revision)


In [ ]:
run('-m', 'pip', 'install', '-r', 'experiments/day11/requirements.txt')


In [ ]:
run('experiments/day12/archive_evidence.py', 'unpack', '--source', 'experiments/day12/run_recovered/evidence/guides', '--out', '/content/day12-guides')


In [ ]:
run('experiments/day12/cache_prompts.py', '--out', '/content/day12-prompts', '--model-cache', '/content/day12-models')


In [ ]:
run('experiments/day12/render.py', '--guides', '/content/day12-guides', '--prompt-dir', '/content/day12-prompts', '--model-cache', '/content/day12-models', '--out', '/content/day12-render')


In [ ]:
run('experiments/day12/assess.py', '--render-root', '/content/day12-render', '--guide-root', '/content/day12-guides', '--output', '/content/day12-assessment.json')


For an interrupted render, append `--resume` to the same command. A changed code/settings/runtime experiment must use a new output directory. Preserve the full render directory, assessment and prompt metadata before disconnecting. GIFs are previews; metrics must use the saved uint8 RGB arrays.